# AgentLab 视频方案 · Colab 一体化部署

**在一个 Colab(A100)里同时跑起:** ComfyUI(GPU 出图/出片/对口型) + AgentLab 编排器(FastAPI 多 Agent) + 前端 UI,只用一个 cloudflared 隧道把 UI 暴露出来。

- **出图 t2i**: FLUX.1-dev (ComfyUI)
- **出片 i2v**: Wan2.2-I2V-14B (GGUF 量化, ComfyUI)
- **对口型 S2V**: Wan2.2-S2V-14B (fp8) + wav2vec2 音频编码 + edge-tts 配音

> ⚠️ **重要前提 / 已知约束**
> 1. **必须 A100(40GB)**。运行时 → 更改运行时类型 → A100。免费 T4(16GB)跑不动满血 Wan2.2-14B。
> 2. **FLUX.1-dev 在 HuggingFace 是 gated 模型**:需先去 https://huggingface.co/black-forest-labs/FLUX.1-dev 点同意协议,并在下面填 `HF_TOKEN`。
> 3. **Colab 是临时环境**:断连后磁盘清空、模型要重下。强烈建议挂载 Google Drive 缓存模型(下面有开关)。
> 4. 本 notebook **未在真实 A100 上端到端验证过**(搭建环境无 GPU)。模型仓库文件名、ComfyUI 节点名偶有上游变动,若某步 404/报节点找不到,按单元格里的提示微调即可。
> 5. RAG(Milvus)在 Colab 不启用,会自动降级——不影响视频功能。

## 0 · 配置区(按需修改)

In [ ]:
#@title 填配置后运行 { display-mode: "form" }

#@markdown ### LLM(对话/编剧大脑,走云 API,不吃 GPU)
OPENAI_API_KEY = ""  #@param {type:"string"}
OPENAI_API_BASE = "https://api.deepseek.com/v1"  #@param {type:"string"}
MODEL_NAME = "deepseek-chat"  #@param {type:"string"}

#@markdown ### HuggingFace Token(下 FLUX.1-dev gated 模型必填)
HF_TOKEN = ""  #@param {type:"string"}

#@markdown ### 仓库地址(默认用你的 fork)
REPO_URL = "https://github.com/aw3n1998/build-with-langchain.git"  #@param {type:"string"}
REPO_BRANCH = "claude/colab-compatibility-deployment-nnakv5"  #@param {type:"string"}

#@markdown ### 用 Google Drive 缓存模型(强烈建议,断连不用重下 ~70GB)
USE_DRIVE_CACHE = True  #@param {type:"boolean"}

#@markdown ### 出片质量(A100 推荐 720*1280 / 81 帧 / 20 步)
VIDEO_SIZE = "720*1280"  #@param ["480*832", "720*1280", "832*480", "768*768"]
VIDEO_FRAMES = 81  #@param {type:"integer"}
VIDEO_FPS = 16  #@param {type:"integer"}
VIDEO_STEPS = 20  #@param {type:"integer"}
S2V_VOICE = "zh-CN-YunxiNeural"  #@param {type:"string"}

import os
assert OPENAI_API_KEY, "请填 OPENAI_API_KEY(DeepSeek/OpenAI 兼容 key)"
print("配置已读取。GPU 检查:")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "⚠️ 没有 GPU!运行时→更改运行时类型→A100"

## 1 · 拉代码 + (可选)挂 Drive 缓存

In [ ]:
import os, subprocess

REPO_DIR = "/content/build-with-langchain"
if not os.path.isdir(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR} || git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

COMFY_DIR = "/content/ComfyUI"
MODELS_ROOT = COMFY_DIR + "/models"

# Drive 缓存:把模型目录放 Drive,断连重连后直接复用
MODELS_CACHE = MODELS_ROOT
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODELS_CACHE = '/content/drive/MyDrive/agentlab_comfy_models'
    os.makedirs(MODELS_CACHE, exist_ok=True)
    print('模型将缓存到:', MODELS_CACHE)
print('REPO_DIR =', REPO_DIR)

## 2 · 安装 ComfyUI + 自定义节点(GGUF / VideoHelperSuite)

In [ ]:
import os
if not os.path.isdir(COMFY_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFY_DIR}
# ComfyUI 依赖(torch 用 Colab 自带的,避免重装)
!pip -q install -r {COMFY_DIR}/requirements.txt

# 自定义节点:GGUF 量化加载 + 视频合成(mp4)
CN = COMFY_DIR + '/custom_nodes'
os.makedirs(CN, exist_ok=True)
for repo in [
    'https://github.com/city96/ComfyUI-GGUF',
    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite',
]:
    name = repo.rsplit('/',1)[-1]
    if not os.path.isdir(f'{CN}/{name}'):
        !git clone {repo} {CN}/{name}
    req = f'{CN}/{name}/requirements.txt'
    if os.path.exists(req):
        !pip -q install -r {req}
print('ComfyUI + 自定义节点就绪')

## 3 · 安装 AgentLab 编排器依赖

去掉 Milvus 相关重依赖也能跑视频(RAG 自动降级),这里按 requirements 全装以保证 import 不缺包。

In [ ]:
!pip -q install -r {REPO_DIR}/requirements.txt python-dotenv huggingface_hub
print('AgentLab 依赖安装完成')

## 4 · 下载模型(~70GB,首次最久;走 Drive 缓存后续秒开)

下载成**与 workflow 模板完全一致的文件名**放进对应目录,模板即可原样工作。某条 404 多半是上游仓库改了路径,按打印的提示去 HF 页面找对应文件名替换即可。

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

# 模型实际落到 Drive 缓存目录,再软链到 ComfyUI/models,断连复用
for sub in ['diffusion_models','unet','clip','vae','audio_encoders']:
    os.makedirs(f'{MODELS_CACHE}/{sub}', exist_ok=True)
    tgt = f'{MODELS_ROOT}/{sub}'
    if MODELS_CACHE != MODELS_ROOT:
        os.makedirs(MODELS_ROOT, exist_ok=True)
        if os.path.islink(tgt) or os.path.exists(tgt):
            if not os.path.islink(tgt): shutil.rmtree(tgt, ignore_errors=True)
        if not os.path.islink(tgt): os.symlink(f'{MODELS_CACHE}/{sub}', tgt)

tok = HF_TOKEN or None

def fetch(repo, filename, sub, rename=None, token=None):
    """下载到缓存目录,按 rename 命名(对齐 workflow 模板)。已存在则跳过。"""
    dst_name = rename or os.path.basename(filename)
    dst = f'{MODELS_CACHE}/{sub}/{dst_name}'
    if os.path.exists(dst) and os.path.getsize(dst) > 1_000_000:
        print('✓ 已存在,跳过:', dst_name); return
    try:
        p = hf_hub_download(repo_id=repo, filename=filename, token=token)
        shutil.copy(p, dst)
        print('✓ 下载完成:', dst_name)
    except Exception as e:
        print(f'✗ 失败 [{repo}/{filename}] -> {dst_name}\n   {type(e).__name__}: {e}\n   去 https://huggingface.co/{repo}/tree/main 找对应文件名替换')

# ── FLUX.1-dev 出图(gated,需 HF_TOKEN)──
fetch('black-forest-labs/FLUX.1-dev', 'flux1-dev.safetensors', 'diffusion_models', token=tok)
fetch('black-forest-labs/FLUX.1-dev', 'ae.safetensors', 'vae', token=tok)
fetch('comfyanonymous/flux_text_encoders', 't5xxl_fp16.safetensors', 'clip')
fetch('comfyanonymous/flux_text_encoders', 'clip_l.safetensors', 'clip')

# ── Wan2.2 公共件(文本编码器 + VAE)──
fetch('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'clip')
fetch('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/vae/wan2.2_vae.safetensors', 'vae')
fetch('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/vae/wan_2.1_vae.safetensors', 'vae')

# ── Wan2.2-I2V-14B GGUF(单专家:用 LowNoise 精修专家)──
fetch('QuantStack/Wan2.2-I2V-A14B-GGUF', 'LowNoise/Wan2.2-I2V-A14B-LowNoise-Q4_K_M.gguf', 'unet', rename='wan2.2-i2v-14B-Q4_K_M.gguf')

# ── Wan2.2-S2V-14B 对口型 + 音频编码器 ──
fetch('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/diffusion_models/wan2.2_s2v_14B_fp8_scaled.safetensors', 'diffusion_models')
fetch('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/audio_encoders/wav2vec2_large_english_fp16.safetensors', 'audio_encoders')

print('\n模型阶段结束。clip 目录也兼作 text_encoders;UNETLoader 会同时扫 diffusion_models/unet。')
!ls -lah {MODELS_CACHE}/diffusion_models {MODELS_CACHE}/unet {MODELS_CACHE}/clip {MODELS_CACHE}/vae {MODELS_CACHE}/audio_encoders 2>/dev/null

## 5 · 写 .env(把 AgentLab 视频后端透明接到本机 ComfyUI)

In [ ]:
i2v_wf = f'{REPO_DIR}/comfyui_workflows/colab/i2v_colab.json'  # 单专家无 SageAttention,出错点少
env = f"""# ── LLM(云 API)──
OPENAI_API_KEY={OPENAI_API_KEY}
OPENAI_API_BASE={OPENAI_API_BASE}
MODEL_NAME={MODEL_NAME}

# ── 视频全链路透明走本机 ComfyUI ──
COMFYUI_BASE_URL=http://127.0.0.1:8188
VIDEO_PROVIDER_DEFAULT=wan2.2
COMFYUI_VIDEO_AS=auto
COMFYUI_IMAGE_AS=auto
COMFYUI_WORKFLOW_I2V={i2v_wf}
# t2i / s2v 用仓库自带模板(文件名已对齐下载)
COMFYUI_S2V_TTS_VOICE={S2V_VOICE}
COMFYUI_SIZE={VIDEO_SIZE}
COMFYUI_FRAMES={VIDEO_FRAMES}
COMFYUI_FPS={VIDEO_FPS}
COMFYUI_STEPS={VIDEO_STEPS}
COMFYUI_T2I_SIZE=768*1024
COMFYUI_T2I_STEPS=28
COMFYUI_TIMEOUT=2400
"""
with open(f'{REPO_DIR}/.env', 'w') as f:
    f.write(env)
print(env)

## 6 · 启动 ComfyUI(后台) → 等待健康

In [ ]:
import subprocess, time, urllib.request

comfy_log = open('/content/comfyui.log','w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
     '--output-directory', '/content/comfy_out'],
    cwd=COMFY_DIR, stdout=comfy_log, stderr=subprocess.STDOUT)

def wait_http(url, name, timeout=300):
    t0=time.time()
    while time.time()-t0 < timeout:
        try:
            urllib.request.urlopen(url, timeout=3); print(f'✓ {name} 就绪'); return True
        except Exception: time.sleep(3)
    print(f'✗ {name} 超时,看日志:'); return False

if not wait_http('http://127.0.0.1:8188/system_stats', 'ComfyUI'):
    !tail -n 40 /content/comfyui.log

## 7 · 启动 AgentLab API(后台) → 等待健康

In [ ]:
import subprocess, os
envp = dict(os.environ)
envp.pop('HF_HUB_OFFLINE', None)  # 允许 FastEmbed 联网下小模型
api_log = open('/content/agentlab_api.log','w')
api_proc = subprocess.Popen(
    ['uvicorn','agent_lab.main_api:app','--host','127.0.0.1','--port','8000'],
    cwd=REPO_DIR, stdout=api_log, stderr=subprocess.STDOUT, env=envp)
if not wait_http('http://127.0.0.1:8000/api/health','AgentLab API', timeout=180):
    !tail -n 50 /content/agentlab_api.log

## 8 · 构建前端 + cloudflared 隧道(拿到公网 UI 地址)

In [ ]:
import os, subprocess
# Node 环境(Colab 自带 node;没有则装)
!node -v || (apt-get -qq update && apt-get -qq install -y nodejs npm)
FE = f'{REPO_DIR}/frontend'
# 让 vite dev 接受 cloudflared 的外部域名
vite_patch = '''import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'
export default defineConfig({
  plugins:[react()],
  server:{ host:true, port:5173, allowedHosts:true,
    proxy:{ '/api':{ target:'http://localhost:8000', changeOrigin:true, ws:true } } },
})
'''
open(f'{FE}/vite.config.js','w').write(vite_patch)
!cd {FE} && npm install --silent
fe_log = open('/content/frontend.log','w')
fe_proc = subprocess.Popen(['npm','run','dev','--','--host'], cwd=FE, stdout=fe_log, stderr=subprocess.STDOUT)
wait_http('http://127.0.0.1:5173','前端 Vite', timeout=180)

In [ ]:
# cloudflared 快速隧道(免登录,随机 trycloudflare 域名)
import subprocess, re, time
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
cf_log = open('/content/cloudflared.log','w+')
cf = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:5173','--no-autoupdate'],
                      stdout=cf_log, stderr=subprocess.STDOUT)
url=None; t0=time.time()
while time.time()-t0 < 60 and not url:
    time.sleep(2); cf_log.seek(0)
    m=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', cf_log.read())
    if m: url=m.group(0)
print('\n================  打开这个地址用 UI  ================')
print('  ', url or '(没抓到,看 /content/cloudflared.log)')
print('====================================================')
print('ComfyUI 本机调试(可选):另开隧道指 http://localhost:8188')

## 9 · 怎么用 / 排错

**用法(在 UI 里):**
1. 打开上面的 trycloudflare 地址。
2. 直接说需求,例如:`把这段小说转成视频:<贴一段文字>` —— video agent 会编排:出分镜 → FLUX 出图 → Wan2.2 让图动起来 → 合成成片。
3. 想让某个镜头**人物开口说话**:打开该镜的「对口型」开关并给一句台词,会自动走 Wan2.2-S2V + edge-tts。
4. 出图/出片的实时进度在对话里流式显示;成片可在线播放/下载。

**排错:**
- 某步报「节点找不到 / class_type」→ 对应自定义节点没装好,回第 2 格重跑;或该模型 GGUF 文件名没对上,看第 4 格提示。
- 出片 OOM → 把第 0 格 `VIDEO_SIZE` 降到 `480*832`、`VIDEO_FRAMES` 降到 49,重跑第 5 格写 .env,再重启第 7 格 API(ComfyUI 不用重启)。
- ComfyUI 没起来 → `!tail -n 60 /content/comfyui.log`。
- API 没起来 → `!tail -n 80 /content/agentlab_api.log`(RAG/Milvus 的 warning 可忽略)。
- 想要更高保真的 i2v(两专家 high/low noise)→ 见 `colab/README.md`。

**保活:** Colab 闲置会回收。出长片时别关页面;模型已缓存到 Drive,重连后第 4 格秒过。

In [ ]:
# 实时看 ComfyUI 出片日志(可选)
!tail -n 40 /content/comfyui.log